# Vector Arm Hexagon

STAGE88: editable controls

In [1]:
import subprocess
import sys
from pathlib import Path

cwd_before = Path.cwd().resolve()
sys_path_before = tuple(sys.path)
import vbb_study as pkg

assert Path(pkg.__file__).is_absolute()
repo_root = Path(pkg.__file__).resolve().parents[1]
assert cwd_before != repo_root
assert all('Publication_Study' not in str(path) for path in sys_path_before)
lock = subprocess.run([sys.executable, '-m', 'pytest', str(repo_root / 'tests' / 'test_characterisation_lock.py')], cwd=repo_root, text=True, capture_output=True)
print(lock.stdout[-1200:])
assert lock.returncode == 0


============================= test session starts =============================
platform win32 -- Python 3.13.7, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\PhD\Code
configfile: pytest.ini
plugins: anyio-4.12.1
collected 9 items

tests\test_characterisation_lock.py .........                            [100%]

============================= 9 passed in 21.28s ==============================



## Section 1 - Configs

In [2]:
import numpy as np
from dataclasses import replace

from vbb_study.design import default_config
from vbb_study.vector_arm_config import SLMPanelConfig, VectorArmConfig
from vbb_study.vector_arm_chain import default_vector_arm_grid, fit_global_phase_error, headless_angle_delta, local_polarization_angle, run_vector_arm
from vbb_study.vector_arm_metrics import assert_degenerate_rejection
from vbb_study.vector_arm_sweeps import delta_rotation_sweep
from vbb_study.vector_axicon import assert_locked_kr_fingerprint, export_surface_field_handoff, resolve_vector_axicon_parameters, run_vector_axicon_to_surface
from vbb_study.vector_field import VectorField, propagate_vector_asm
from vbb_study.vector_fourier import carrier_collinearity_report
from vbb_study.viz_fields import _phase_winding

cfg = VectorArmConfig(ideal_components=True)
assert cfg.wavelength_m == 1029e-9
assert cfg.pulse_duration_s == 260e-15
assert cfg.waist_m == 2e-3
assert cfg.slm1.n_x == 1920 and cfg.slm1.n_y == 1080
assert cfg.slm1.pitch_m == 8e-6
assert cfg.slm1.phase_levels == 256
assert cfg.slm1.fill_factor == 0.93


## Sections 2-5 - Chain Assertions

In [3]:
grid = default_vector_arm_grid(cfg, n=256)
run = run_vector_arm(cfg, grid=grid, return_debug=True)
gamma, err = fit_global_phase_error(run.field, run.target)
print('ideal reconstruction error', err)
assert err < 1e-10

wrong = run_vector_arm(cfg, grid=grid, naive_psi2=True, return_debug=True)
_, wrong_err = fit_global_phase_error(wrong.field, wrong.target)
print('naive psi2 reconstruction error', wrong_err)
assert wrong_err > 0.1

e_plus, e_minus = run.field.circular_components()
w_plus = _phase_winding(e_plus, grid, cfg.waist_m / 2.0, n_phi=1024)
w_minus = _phase_winding(e_minus, grid, cfg.waist_m / 2.0, n_phi=1024)
print(f'winding plus={w_plus:.12f} minus={w_minus:.12f}')
assert abs(w_plus + 1.0) < 1e-9
assert abs(w_minus - 1.0) < 1e-9

radial = run_vector_arm(cfg, grid=grid, all_radial=True)
psi = local_polarization_angle(radial)
ring = (grid['R'] > 0.45 * cfg.waist_m) & (grid['R'] < 0.55 * cfg.waist_m)
radial_rms = float(np.sqrt(np.mean(headless_angle_delta(psi[ring], grid['PHI'][ring]) ** 2)))
stokes = radial.stokes()
print('radial rms', radial_rms)
assert radial_rms < 1e-9
assert np.nanmax(np.abs(stokes['S3'])) < 1e-10 * np.nanmax(stokes['S0'])

rows = delta_rotation_sweep(cfg, deltas=[0.2, 0.7, 1.4], grid=grid)
print(rows)
assert all(abs(row['error_rad']) < 1e-3 for row in rows)


ideal reconstruction error 5.649708068118573e-16
naive psi2 reconstruction error 0.7071067811865477
winding plus=-1.000000000000 minus=1.000000000000
radial rms 1.7398898069206842e-16
[{'delta_rad': 0.2, 'measured_rotation_rad': 0.10000000000000003, 'expected_rotation_rad': 0.1, 'error_rad': 2.7755575615628914e-17}, {'delta_rad': 0.7, 'measured_rotation_rad': 0.35000000000000003, 'expected_rotation_rad': 0.35, 'error_rad': 5.551115123125783e-17}, {'delta_rad': 1.4, 'measured_rotation_rad': 0.7000000000000001, 'expected_rotation_rad': 0.7, 'error_rad': 1.1102230246251565e-16}]


In [4]:
real_cfg = VectorArmConfig(slm1=SLMPanelConfig(carrier_lp_per_mm=1.5625, carrier_sign=1), slm2=SLMPanelConfig(carrier_lp_per_mm=1.5625, carrier_sign=-1))
real_grid = default_vector_arm_grid(real_cfg, n=256)
real_run = run_vector_arm(real_cfg, grid=real_grid, return_debug=True)
rp, rm = real_run.field.circular_components()
report = carrier_collinearity_report(rp, rm, real_grid)
print('carrier separation pixels', report.separation_pixels)
assert report.separation_pixels < 0.5
wrong_carrier_cfg = replace(real_cfg, slm2=replace(real_cfg.slm2, carrier_sign=+1))
wrong_carrier = run_vector_arm(wrong_carrier_cfg, grid=real_grid)
wp, wm = wrong_carrier.circular_components()
wrong_report = carrier_collinearity_report(wp, wm, real_grid)
expected_sep = 2.0 * abs(real_cfg.slm1.carrier_lp_per_m)
print('wrong carrier separation cpm', wrong_report.separation_cpm)
assert abs(wrong_report.separation_cpm - expected_sep) / expected_sep < 0.05
assert real_run.slm1 is not None and real_run.slm1.ledger.relative_error < 1e-12
assert real_run.slm2 is not None and real_run.slm2.ledger.relative_error < 1e-12


carrier separation pixels 0.029081261107556593
wrong carrier separation cpm 3074.6820671291102


## Section 2 - Vector ASM Energy

In [5]:
g = default_vector_arm_grid(cfg, n=128)
gauss = np.exp(-(g['R'] / (0.4 * cfg.waist_m)) ** 2).astype(complex)
vf0 = propagate_vector_asm(VectorField(gauss, np.zeros_like(gauss), grid=g, wavelength_m=cfg.wavelength_m), 0.0)
vf1 = propagate_vector_asm(vf0, 1e-6)
assert abs(vf1.power / vf0.power - 1.0) < 1e-12
checker = ((-1.0) ** (np.indices(gauss.shape).sum(axis=0))).astype(complex)
ev0 = propagate_vector_asm(VectorField(checker, np.zeros_like(checker), grid=g, wavelength_m=cfg.wavelength_m), 0.0)
ev1 = propagate_vector_asm(ev0, 1e-6)
assert ev1.power <= ev0.power + 1e-18


## Sections 6-8 - Surface Handoff and Metrics

In [6]:
twin = default_config('fast')
params = resolve_vector_axicon_parameters(twin)
assert_locked_kr_fingerprint(params.k_r_surface_m_inv)
surface_result = run_vector_axicon_to_surface(run.field, twin, z_values_m=[-1e-6, 0.0, 1e-6])
assert surface_result.surface.medium_before == 1.0
assert surface_result.intensity_stack.shape[0] == 3
degenerate = assert_degenerate_rejection(0.1)
print(degenerate)


{'ring_h6': 0.0, 'lattice_h6': 0.002933488763299709, 'ring_lattice_ratio': 0.0, 'lattice_artifact_ratio': 0.9999937250892786}


## Limitations and Known Omissions

- No inter-pixel phase crosstalk.
- No SLM flicker or temporal phase noise.
- Thin-element axicon, with no thick-element walk-off.
- Collimated-incidence Fresnel angles at the axicon, not per-ray angles.
- Vector ASM rather than a Richards-Wolf focal model at NA 0.45.
- In-medium propagation is out of scope for this notebook.

In [7]:
cone_half_angle = float(np.arcsin(params.k_r_surface_m_inv / (2.0 * np.pi / twin.laser.wavelength_m)))
print('focal cone half-angle rad', cone_half_angle, 'deg', np.rad2deg(cone_half_angle))
git = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo_root, text=True, capture_output=True)
git_sha = git.stdout.strip() if git.returncode == 0 else 'unknown'
paths = export_surface_field_handoff(surface_result.surface, repo_root / 'outputs' / 'stage7_vector_arm' / 'surface_field_stage7.npz', config=cfg, git_sha=git_sha, lock_status='fast_lock_green')
print(paths)
final_lock = subprocess.run([sys.executable, '-m', 'pytest', str(repo_root / 'tests' / 'test_characterisation_lock.py')], cwd=repo_root, text=True, capture_output=True)
print(final_lock.stdout[-1200:])
assert final_lock.returncode == 0


focal cone half-angle rad 0.2656936067432321 deg 15.22312230999583
{'npz': WindowsPath('C:/PhD/Code/Publication_Study/outputs/stage7_vector_arm/surface_field_stage7.npz'), 'sidecar': WindowsPath('C:/PhD/Code/Publication_Study/outputs/stage7_vector_arm/surface_field_stage7.npz.json')}
============================= test session starts =============================
platform win32 -- Python 3.13.7, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\PhD\Code
configfile: pytest.ini
plugins: anyio-4.12.1
collected 9 items

tests\test_characterisation_lock.py .........                            [100%]

============================= 9 passed in 18.87s ==============================

